In [8]:
###Cargar el Dataset

!pip install kagglehub

import kagglehub
path = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/martj42/international-football-results-from-1872-to-2017/versions/129


In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FootballResults") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/02 02:16:07 INFO SparkEnv: Registering MapOutputTracker
26/07/02 02:16:07 INFO SparkEnv: Registering BlockManagerMaster
26/07/02 02:16:07 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/07/02 02:16:07 INFO SparkEnv: Registering OutputCommitCoordinator


In [24]:
import os
local_path = os.path.join(path, 'results.csv')

!hdfs dfs -mkdir -p /user/root/data
!hdfs dfs -put -f {local_path} /user/root/data/results.csv

In [25]:
!hdfs dfs -ls /user/root/data/

Found 1 items
-rw-r--r--   2 root hadoop    3725571 2026-07-02 02:43 /user/root/data/results.csv


In [26]:
results = spark.read.csv("hdfs:///user/root/data/results.csv", header=True, inferSchema=True)
results.show(5)
results.printSchema()

+----------+---------+---------+----------+----------+----------+-------+--------+-------+
|      date|home_team|away_team|home_score|away_score|tournament|   city| country|neutral|
+----------+---------+---------+----------+----------+----------+-------+--------+-------+
|1872-11-30| Scotland|  England|         0|         0|  Friendly|Glasgow|Scotland|  false|
|1873-03-08|  England| Scotland|         4|         2|  Friendly| London| England|  false|
|1874-03-07| Scotland|  England|         2|         1|  Friendly|Glasgow|Scotland|  false|
|1875-03-06|  England| Scotland|         2|         2|  Friendly| London| England|  false|
|1876-03-04| Scotland|  England|         3|         0|  Friendly|Glasgow|Scotland|  false|
+----------+---------+---------+----------+----------+----------+-------+--------+-------+
only showing top 5 rows

root
 |-- date: date (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_score: string (nullable =

In [27]:
###REvisar que la columna date este en el formato correcto
results.select("date").show(5, truncate=False)
results.printSchema()

+----------+
|date      |
+----------+
|1872-11-30|
|1873-03-08|
|1874-03-07|
|1875-03-06|
|1876-03-04|
+----------+
only showing top 5 rows

root
 |-- date: date (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_score: string (nullable = true)
 |-- away_score: string (nullable = true)
 |-- tournament: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- neutral: boolean (nullable = true)



In [28]:
###Confirmar cuales fueron los ultimos resultados cargados al dataset
results.tail(20)

[Row(date=datetime.date(2026, 6, 27), home_team='DR Congo', away_team='Uzbekistan', home_score='3', away_score='1', tournament='FIFA World Cup', city='Atlanta', country='United States', neutral=True),
 Row(date=datetime.date(2026, 6, 27), home_team='Panama', away_team='England', home_score='0', away_score='2', tournament='FIFA World Cup', city='East Rutherford', country='United States', neutral=True),
 Row(date=datetime.date(2026, 6, 27), home_team='Croatia', away_team='Ghana', home_score='2', away_score='1', tournament='FIFA World Cup', city='Philadelphia', country='United States', neutral=True),
 Row(date=datetime.date(2026, 6, 28), home_team='South Africa', away_team='Canada', home_score='0', away_score='1', tournament='FIFA World Cup', city='Inglewood', country='United States', neutral=True),
 Row(date=datetime.date(2026, 6, 29), home_team='Brazil', away_team='Japan', home_score='2', away_score='1', tournament='FIFA World Cup', city='Houston', country='United States', neutral=True)

In [29]:
##Actualizar el df

from pyspark.sql import functions as F

corrections = [
    ("2026-06-30", "Ivory Coast", "Norway", "1", "2"),
    ("2026-06-30", "France", "Sweden", "3", "0"),
    ("2026-06-30", "Mexico", "Ecuador", "2", "0"),
    ("2026-07-01", "England", "DR Congo", "2", "1"),
    ("2026-07-01", "Belgium", "Senegal", "3", "2"),
    ("2026-07-01", "United States", "Bosnia and Herzegovina", "2", "0"),
]

results_updated = results

for match_date, home, away, hs, as_ in corrections:
    results_updated = results_updated.withColumn(
        "home_score",
        F.when(
            (F.col("date") == F.to_date(F.lit(match_date))) &
            (F.col("home_team") == home) &
            (F.col("away_team") == away),
            F.lit(hs)
        ).otherwise(F.col("home_score"))
    ).withColumn(
        "away_score",
        F.when(
            (F.col("date") == F.to_date(F.lit(match_date))) &
            (F.col("home_team") == home) &
            (F.col("away_team") == away),
            F.lit(as_)
        ).otherwise(F.col("away_score"))
    )

# Verificar la actualización
results_updated.filter(
    F.col("date").isin("2026-06-30", "2026-07-01")
).show(truncate=False)

+----------+-------------+----------------------+----------+----------+--------------+---------------+-------------+-------+
|date      |home_team    |away_team             |home_score|away_score|tournament    |city           |country      |neutral|
+----------+-------------+----------------------+----------+----------+--------------+---------------+-------------+-------+
|2026-06-30|Ivory Coast  |Norway                |1         |2         |FIFA World Cup|Arlington      |United States|true   |
|2026-06-30|France       |Sweden                |3         |0         |FIFA World Cup|East Rutherford|United States|true   |
|2026-06-30|Mexico       |Ecuador               |2         |0         |FIFA World Cup|Mexico City    |Mexico       |false  |
|2026-07-01|England      |DR Congo              |2         |1         |FIFA World Cup|Atlanta        |United States|true   |
|2026-07-01|Belgium      |Senegal               |3         |2         |FIFA World Cup|Seattle        |United States|true   |


In [31]:
results_2000 = results_updated.filter(F.col("date") >= "2000-01-01")

print("Total filas:", results.count())
print("Resultados a partir del 2000-01-01:", results_filtered.count())

results_2000.show(20)

Total filas: 49494
Resultados a partir del 2000-01-01: 25432
+----------+-------------------+-------------------+----------+----------+----------+-------------+-------------------+-------+
|      date|          home_team|          away_team|home_score|away_score|tournament|         city|            country|neutral|
+----------+-------------------+-------------------+----------+----------+----------+-------------+-------------------+-------+
|2000-01-04|              Egypt|               Togo|         2|         1|  Friendly|        Aswan|              Egypt|  false|
|2000-01-07|            Tunisia|               Togo|         7|         0|  Friendly|        Tunis|            Tunisia|  false|
|2000-01-08|Trinidad and Tobago|             Canada|         0|         0|  Friendly|Port of Spain|Trinidad and Tobago|  false|
|2000-01-09|       Burkina Faso|              Gabon|         1|         1|  Friendly|  Ouagadougou|       Burkina Faso|  false|
|2000-01-09|          Guatemala|           

In [32]:
##Modelo de decaimiento exponencial para el df

# Fecha de referencia 01Jul2026
reference_date = results_2000.agg(F.max("date")).collect()[0][0]

# Dias desde el partido
results_weighted = results_2000.withColumn(
    "days_ago", F.datediff(F.lit(reference_date), F.col("date"))
)

# Exponential decay weight: weight = exp(-days_ago / half_life_days)
#Se usara un half-life corto para que los resultados recientes tengan más impacto

half_life_days = 365

results_weighted = results_weighted.withColumn(
    "weight", F.exp(-F.col("days_ago") / F.lit(half_life_days) * F.log(F.lit(2.0)))
)

results_weighted.select("date", "home_team", "away_team", "days_ago", "weight").show(10)

+----------+-------------------+---------+--------+--------------------+
|      date|          home_team|away_team|days_ago|              weight|
+----------+-------------------+---------+--------+--------------------+
|2000-01-04|              Egypt|     Togo|    9678|1.042723224063107E-8|
|2000-01-07|            Tunisia|     Togo|    9675|1.048680676650507...|
|2000-01-08|Trinidad and Tobago|   Canada|    9674|1.050674048392073...|
|2000-01-09|       Burkina Faso|    Gabon|    9673|1.052671209209747...|
|2000-01-09|          Guatemala|  Armenia|    9673|1.052671209209747...|
|2000-01-09|        Ivory Coast|    Egypt|    9673|1.052671209209747...|
|2000-01-09|             Mexico|     Iran|    9673|1.052671209209747...|
|2000-01-11|            Bermuda|   Canada|    9671|1.056676926896800...|
|2000-01-11|       Burkina Faso| Cameroon|    9671|1.056676926896800...|
|2000-01-13|            Senegal| Cameroon|    9669|1.060697887495456...|
+----------+-------------------+---------+--------+

In [33]:
from pyspark.sql.window import Window

#Asignar IDs
results_id = results_weighted.withColumn("match_id", F.monotonically_increasing_id())

# Crear team perspective
home_rows = results_id.select(
    "match_id", "date",
    F.col("home_team").alias("team"),
    F.col("home_score").cast("int").alias("goals_scored"),
    F.col("away_score").cast("int").alias("goals_conceded"),
    F.when(F.col("neutral") == True, "neutral").otherwise("home").alias("venue")
)
away_rows = results_id.select(
    "match_id", "date",
    F.col("away_team").alias("team"),
    F.col("away_score").cast("int").alias("goals_scored"),
    F.col("home_score").cast("int").alias("goals_conceded"),
    F.when(F.col("neutral") == True, "neutral").otherwise("away").alias("venue")
)
team_matches = home_rows.unionByName(away_rows).filter(F.col("goals_scored").isNotNull())

# Self-join:
half_life_days = 365

t1 = team_matches.alias("t1")
t2 = team_matches.alias("t2")

prior_matches = (
    t1.join(t2, (F.col("t1.team") == F.col("t2.team")) & (F.col("t2.date") < F.col("t1.date")))
    .withColumn("days_gap", F.datediff(F.col("t1.date"), F.col("t2.date")))
    .withColumn("w", F.exp(-F.col("days_gap") / F.lit(half_life_days) * F.log(F.lit(2.0))))
    .groupBy(F.col("t1.match_id").alias("match_id"), F.col("t1.team").alias("team"))
    .agg(
        F.sum(F.col("t2.goals_scored") * F.col("w")).alias("num_scored"),
        F.sum(F.col("t2.goals_conceded") * F.col("w")).alias("num_conceded"),
        F.sum("w").alias("den"),
        F.count("*").alias("prior_matches_count")
    )
    .withColumn("pre_avg_scored", F.col("num_scored") / F.col("den"))
    .withColumn("pre_avg_conceded", F.col("num_conceded") / F.col("den"))
    .select("match_id", "team", "pre_avg_scored", "pre_avg_conceded", "prior_matches_count")
)

In [34]:
home_features = prior_matches.select(
    F.col("match_id"),
    F.col("team").alias("home_team"),
    F.col("pre_avg_scored").alias("home_pre_avg_scored"),
    F.col("pre_avg_conceded").alias("home_pre_avg_conceded"),
    F.col("prior_matches_count").alias("home_prior_count")
)
away_features = prior_matches.select(
    F.col("match_id"),
    F.col("team").alias("away_team"),
    F.col("pre_avg_scored").alias("away_pre_avg_scored"),
    F.col("pre_avg_conceded").alias("away_pre_avg_conceded"),
    F.col("prior_matches_count").alias("away_prior_count")
)

model_data = (
    results_id
    .join(home_features, ["match_id", "home_team"])
    .join(away_features, ["match_id", "away_team"])
    .filter((F.col("home_prior_count") >= 5) & (F.col("away_prior_count") >= 5))  # drop teams with too little history
    .withColumn("home_score", F.col("home_score").cast("int"))
    .withColumn("away_score", F.col("away_score").cast("int"))
    .filter(F.col("home_score").isNotNull() & F.col("away_score").isNotNull())
)

model_data.select(
    "date", "home_team", "away_team", "home_score", "away_score",
    "home_pre_avg_scored", "home_pre_avg_conceded",
    "away_pre_avg_scored", "away_pre_avg_conceded", "neutral"
).show(10)

+----------+----------+-----------+----------+----------+-------------------+---------------------+-------------------+---------------------+-------+
|      date| home_team|  away_team|home_score|away_score|home_pre_avg_scored|home_pre_avg_conceded|away_pre_avg_scored|away_pre_avg_conceded|neutral|
+----------+----------+-----------+----------+----------+-------------------+---------------------+-------------------+---------------------+-------+
|2000-06-28|   Tunisia|    Algeria|         2|         2| 1.9593744363065584|    1.226207577608289| 1.4593496031763977|   0.5678731780285611|  false|
|2002-09-24|   Algeria|     Uganda|         1|         1| 1.2961820923279375|   1.1851840396835902|  1.511497331549798|   1.2396259401280416|  false|
|2003-06-07|   Finland|     Serbia|         3|         0| 1.2806634397153975|    1.050376883078457| 1.3041633631432974|   1.3886842542231435|  false|
|2004-06-20|     Egypt|Ivory Coast|         1|         2| 1.9051500670300598|   0.7551301310745194| 

In [35]:
### Regresión de Poisson

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GeneralizedLinearRegression

# Dependencia de goles en casa: home attack, away defense, neutral flag
assembler_home = VectorAssembler(
    inputCols=["home_pre_avg_scored", "away_pre_avg_conceded", "neutral"],
    outputCol="features"
)
home_train = assembler_home.transform(model_data).select("features", F.col("home_score").alias("label"))

glr_home = GeneralizedLinearRegression(family="poisson", link="log", maxIter=50)
home_model = glr_home.fit(home_train)

print("Home goals model coefficients:", home_model.coefficients)
print("Home goals model intercept:", home_model.intercept)
print(home_model.summary)

26/07/02 02:59:12 WARN Instrumentation: [0cdc6493] regParam is zero, which might cause numerical instability and overfitting.
26/07/02 02:59:14 WARN Instrumentation: [0cdc6493] regParam is zero, which might cause numerical instability and overfitting.
26/07/02 02:59:15 WARN Instrumentation: [0cdc6493] regParam is zero, which might cause numerical instability and overfitting.
26/07/02 02:59:15 WARN Instrumentation: [0cdc6493] regParam is zero, which might cause numerical instability and overfitting.
26/07/02 02:59:15 WARN Instrumentation: [0cdc6493] regParam is zero, which might cause numerical instability and overfitting.
26/07/02 02:59:15 WARN Instrumentation: [0cdc6493] regParam is zero, which might cause numerical instability and overfitting.
26/07/02 02:59:18 WARN Instrumentation: [0cdc6493] regParam is zero, which might cause numerical instability and overfitting.
26/07/02 02:59:20 WARN Instrumentation: [0cdc6493] regParam is zero, which might cause numerical instability and overf

Home goals model coefficients: [0.28187754881216937,0.19840094889152413,-0.18075972795918707]
Home goals model intercept: -0.17215025920236668


Coefficients:
             Feature Estimate Std Error  T Value P Value
         (Intercept)  -0.1722    0.0125 -13.7749  0.0000
 home_pre_avg_scored   0.2819    0.0074  38.0913  0.0000
away_pre_avg_conc...   0.1984    0.0028  70.5831  0.0000
             neutral  -0.1808    0.0116 -15.5764  0.0000

(Dispersion parameter for poisson family taken to be 1.0000)
    Null deviance: 39139.7641 on 24347 degrees of freedom
Residual deviance: 34807.8114 on 24347 degrees of freedom
AIC: 80424.7025


In [36]:
# Dependencia goles visitante: away attack, home defense, neutral flag
assembler_away = VectorAssembler(
    inputCols=["away_pre_avg_scored", "home_pre_avg_conceded", "neutral"],
    outputCol="features"
)
away_train = assembler_away.transform(model_data).select("features", F.col("away_score").alias("label"))

glr_away = GeneralizedLinearRegression(family="poisson", link="log", maxIter=50)
away_model = glr_away.fit(away_train)

print("Away goals model coefficients:", away_model.coefficients)
print("Away goals model intercept:", away_model.intercept)

26/07/02 03:01:20 WARN Instrumentation: [d9a4ded0] regParam is zero, which might cause numerical instability and overfitting.
26/07/02 03:01:23 WARN Instrumentation: [d9a4ded0] regParam is zero, which might cause numerical instability and overfitting.
26/07/02 03:01:23 WARN Instrumentation: [d9a4ded0] regParam is zero, which might cause numerical instability and overfitting.
26/07/02 03:01:26 WARN Instrumentation: [d9a4ded0] regParam is zero, which might cause numerical instability and overfitting.
26/07/02 03:01:26 WARN Instrumentation: [d9a4ded0] regParam is zero, which might cause numerical instability and overfitting.
26/07/02 03:01:26 WARN Instrumentation: [d9a4ded0] regParam is zero, which might cause numerical instability and overfitting.
26/07/02 03:01:29 WARN Instrumentation: [d9a4ded0] regParam is zero, which might cause numerical instability and overfitting.


Away goals model coefficients: [0.2942571079906697,0.2962003607796658,0.16661259802673423]
Away goals model intercept: -0.7915634400408119


In [37]:
###Predecir Mexico vs Inglaterra


# Ultimos resultados para cada equipo
latest_per_team = (
    prior_matches
    .join(results_id.select("match_id", "date"), "match_id")
    .withColumn("rn", F.row_number().over(Window.partitionBy("team").orderBy(F.col("date").desc())))
    .filter(F.col("rn") == 1)
    .select("team", "pre_avg_scored", "pre_avg_conceded", "prior_matches_count")
)

latest_per_team.filter(F.col("team").isin("Mexico", "England")).show(truncate=False)

+-------+------------------+------------------+-------------------+
|team   |pre_avg_scored    |pre_avg_conceded  |prior_matches_count|
+-------+------------------+------------------+-------------------+
|England|2.068477141700438 |0.5335747646249225|321                |
|Mexico |1.5722559634256457|0.722347562475893 |476                |
+-------+------------------+------------------+-------------------+



In [38]:
from pyspark.sql import Row

# Estadisticas previas al partido
mex_avg_scored, mex_avg_conceded = 1.5722559634256457, 0.722347562475893
eng_avg_scored, eng_avg_conceded = 2.068477141700438, 0.5335747646249225

# --- Goles esperados por México (casa) ---
sample_home = spark.createDataFrame([
    Row(home_pre_avg_scored=mex_avg_scored, away_pre_avg_conceded=eng_avg_conceded, neutral=0)
])
sample_home_vec = assembler_home.transform(sample_home)
mex_xg = home_model.transform(sample_home_vec).select("prediction").collect()[0]["prediction"]

# --- Goles esperados por Inglaterra (visitante) ---
sample_away = spark.createDataFrame([
    Row(away_pre_avg_scored=eng_avg_scored, home_pre_avg_conceded=mex_avg_conceded, neutral=0)
])
sample_away_vec = assembler_away.transform(sample_away)
eng_xg = away_model.transform(sample_away_vec).select("prediction").collect()[0]["prediction"]

print(f"Goles esperados por México (casa): {mex_xg:.3f}")
print(f"Goles esperados por Inglaterra (visitante) {eng_xg:.3f}")

Goles esperados por México (casa): 1.458
Goles esperados por Inglaterra (visitante) 1.032
